# Install Dependencies
You will need to restart the kernel after the first load

## BLAST+
The example model requires BLAST+, NCBI’s sequence search program. BLAST+ is a compiled program, not a Python package, so a script from the repository installs it. It’s installed once per disk, and this cell makes it visible to the notebook.    
Install via the command line once, then add to the notebook PATH

```bash ~/blast-proticelli/training/install_blast.sh ~/tools```

In [ ]:
os.environ["PATH"] = str(Path.home() / "tools" / "ncbi-blast-2.17.0+" / "bin") + os.pathsep + os.environ["PATH"]
print(subprocess.run(["blastp", "-version"], capture_output=True, text=True).stdout)

## cm4ai-evaluate
Current evaluation metrics

In [ ]:
%pip install "cm4ai-evaluate[extended] @ git+https://github.com/cm4ai-anvil-dream-challenge/cm4ai-evaluate.git"

## Reference data for BLAST

BLAST identifies each protein by searching its sequence against reference data: the reviewed human proteins from UniProt (`human_blast_db.zip`) and a map from protein IDs to gene names (`human_gene_map.zip`). These are built by `training/build_human_blast_db.sh` and `training/build_human_gene_map.sh`.

They are kept in two places:

* **The workspace bucket** holds the master copy. It survives deleting the persistent disk, a new workspace can copy it in one command, and workflows can only read files from a bucket.
* **The persistent disk** (`~/reference/`) holds the working copy the notebook reads. Anything outside `/home/jupyter` is lost when the cloud environment is deleted, and `~/reference/` sits outside the repository clone, so it never ends up in git.

**Once, from a local terminal:** upload both files to the workspace bucket. The bucket name is on the workspace Dashboard. The account signed in to `gcloud` must have access to the workspace.

    gcloud auth list
    gcloud storage cp human_blast_db.zip human_gene_map.zip gs://fc-XXXX/reference/

The next cell copies them to the persistent disk, once.

In [ ]:
bucket = os.environ["WORKSPACE_BUCKET"]          # set by AnVIL
for zip_path in (BLAST_DB_ZIP, GENE_MAP_ZIP):
    if not zip_path.exists():
        zip_path.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["gcloud", "storage", "cp", f"{bucket}/reference/{zip_path.name}", str(zip_path)], check=True)
    print(zip_path, f"{zip_path.stat().st_size / 1e6:.0f} MB")

# Settings
Point the notebook at a training run that exists on the same persistent disk. Match the data set, run mode and seed spedified in the training notebook.  
Download 

In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path
import pandas as pd
import numpy as np

DATASET, RUN_MODE, SEED = "KOLF-train_v1.0", "smoke", 1     # must match selected training run on persistent disk
PREDICTION_STEPS = 20       # diffusion steps per prediction; about 5 s per cell on a T4

REPO = Path.home() / "blast-proticelli"
DATA_DIR = Path.home() / "data" / DATASET
WEIGHTS = Path.home() / "reference" / "proticelli"                    # published (starting) weights
RUN_NAME = f"{DATASET}_{RUN_MODE}_seed{SEED}"
OUT_DIR = Path.home() / "runs" / RUN_NAME
TRAINED_ZIP = Path.home() / "weights" / f"trained_proticelli_{RUN_NAME}.zip"
BLAST_DB_ZIP = Path.home() / "reference" / "human_blast_db.zip"       # adjust to where these were copied
GENE_MAP_ZIP = Path.home() / "reference" / "human_gene_map.zip"
PRED_DIR = OUT_DIR / "prediction"
PRED_DIR.mkdir(parents=True, exist_ok=True)

# Assemble Prediction Weights
Compare prediction across two specified weights models and package them with model specific resources. 

In [ ]:
def assemble(model_dir, zip_path):
    zip_path.unlink(missing_ok=True)
    proc = subprocess.run(["bash", str(REPO / "training" / "build_weights_zip.sh"), str(model_dir),
                           str(BLAST_DB_ZIP), str(GENE_MAP_ZIP), str(zip_path)], capture_output=True, text=True)
    print(proc.stderr or f"{zip_path} ({zip_path.stat().st_size / 2**30:.1f} GiB)")

STARTING = Path.home() / "weights" / "blast_proticelli_starting.zip"
if not STARTING.exists():
    assemble(WEIGHTS, STARTING)

TRAINED = Path.home() / "weights" / f"blast_proticelli_trained_{RUN_NAME}.zip"
if not TRAINED.exists() or TRAINED.stat().st_mtime < TRAINED_ZIP.stat().st_mtime:   # missing, or older than the training run
    unpacked = PRED_DIR / "trained_proticelli"
    shutil.unpack_archive(TRAINED_ZIP, unpacked)
    assemble(unpacked, TRAINED)
    shutil.rmtree(unpacked)

# Predict the held-out cells

Predict every held-out cell with both weights files.   
For each cell, three reference images and the protein’s amino acid sequence go in, and one predicted protein image comes out.   
The protein’s name is never given: the model identifies it from the sequence.

In [ ]:
test = pd.read_csv(OUT_DIR / "cells.csv", dtype=str).query("split == 'test'")
records = [{"prediction_id": r["image_key"],
            "microtubules_image": str(DATA_DIR / r["microtubules_path"]),
            "nucleus_image": str(DATA_DIR / r["nucleus_path"]),
            "er_image": str(DATA_DIR / r["er_path"]),
            "fasta_file": str(DATA_DIR / r["sequence_path"])} for _, r in test.iterrows()]
inputs_json = PRED_DIR / "inputs.json"
inputs_json.write_text(json.dumps(records, indent=2))
print(f"{len(records)} held-out cells, about {len(records) * 5 / 60:.0f} minutes per weights file")

for name, weights_zip in [("starting", STARTING), ("trained", TRAINED)]:
    out = PRED_DIR / name
    if (out / "prediction_results.tsv").exists():
        print(f"{name}: already predicted, in {out}")
        continue
    proc = subprocess.run([sys.executable, str(REPO / "run_inference.py"), "--inputs", str(inputs_json),
                           "--weights_zip", str(weights_zip), "--output_dir", str(out),
                           "--weights_dir", str(PRED_DIR / "unpacked" / name / "weights"),
                           "--reference_data_dir", str(PRED_DIR / "unpacked" / name / "reference"),
                           "--num_inference_steps", str(PREDICTION_STEPS), "--seed", str(SEED)],
                          capture_output=True, text=True)
    print(f"{name}: exit code {proc.returncode}, predictions in {out}")
    if proc.returncode:
        print(proc.stdout[-1500:], proc.stderr[-1500:])

# Evaluate the predictions

Score both sets of predictions against the real masked protein images using cm4ai-evaluate, the challenge evaluation tool. The first table shows each metric averaged over all held-out cells, which direction is better, and the difference training made. The second shows one metric per protein. The challenge metrics are still being refined, so these numbers demonstrate the evaluation step.

Score both sets of predictions against the ground truth protein images with cm4ai-evaluate, the challenge evaluation tool. Each held-out cell is scored separately, and per-cell scores are saved in the evaluation folder. This takes a minute or two and only needs rerunning when the predictions change.

In [ ]:
EVALUATE = (shutil.which("cm4ai-evaluate")
            or next((str(p) for p in [Path(sys.executable).parent / "cm4ai-evaluate",
                                      Path.home() / ".local" / "bin" / "cm4ai-evaluate"] if p.exists()), None))
assert EVALUATE, "cm4ai-evaluate is not installed in this notebook's Python"
EVAL_DIR = PRED_DIR / "evaluation"
EVAL_DIR.mkdir(exist_ok=True)

# Score each set of predictions
scores = {}
for name in ("starting", "trained"):
    manifest_csv = EVAL_DIR / f"{name}_manifest.csv"
    pd.DataFrame({"prediction_id": test["image_key"],
                  "prediction": [str(PRED_DIR / name / f"{k}.png") for k in test["image_key"]],
                  "ground_truth": [str(DATA_DIR / p) for p in test["protein_path"]],
                  "mask": [str(DATA_DIR / p) for p in test["mask_path"]],
                  "group_id": test["antibody_id"]}).to_csv(manifest_csv, index=False)
    subprocess.run([EVALUATE, str(manifest_csv), "-o", str(EVAL_DIR / name), "--metrics", "basic,extended"],
                   capture_output=True)
    scores[name] = pd.read_csv(EVAL_DIR / f"{name}.csv").set_index("prediction_id")

Summarize performance

Every metric computed inside the cell masks, as the challenge scores predictions, averaged over the held-out cells. The table also shows the direction that counts as better and the share of cells where training helped. A second table shows Pearson r per protein. 

The building-block sums (*_sum) are interim values other metrics are calculated from. They grow with cell size, so they have no better direction across different cells, and cells improved is left blank for them. On a fixed set of cells, they become comparable: scored after each epoch, the error sums track total error, and prediction_square_sum shows whether predictions are getting brighter or dimmer as training goes on.

In [ ]:
# All masked metrics, averaged over the held-out cells
DIRECTION = {"maximum_absolute_error": "lower", "mean_absolute_error": "lower", "root_mean_squared_error": "lower",
             "cosine_similarity": "higher", "pearson_r": "higher", "spearman_r": "higher", "psnr_db": "higher",
             "ms_ssim": "higher", "mutual_information_nats": "higher", "haralick_similarity": "higher",
             "gabor_similarity": "higher", "brightness_spread_w1": "lower"}
start, trained = (scores[n][[c for c in scores[n].columns if c.startswith("masked_")]]
                  .select_dtypes("number").replace([np.inf, -np.inf], np.nan) for n in ("starting", "trained"))
rows = []
for m in start.columns:
    direction = DIRECTION.get(m.removeprefix("masked_"), "neither")
    diff = trained[m] - start[m]
    improved = (diff > 0) if direction == "higher" else (diff < 0) if direction == "lower" else None
    rows.append({"metric": m, "better": direction, "starting": start[m].mean(), "trained": trained[m].mean(),
                 "trained minus starting": diff.mean(),
                 "cells improved": improved[diff.notna()].mean() if improved is not None else np.nan})
table = pd.DataFrame(rows).set_index("metric")
display(table.style.format(precision=3).format(na_rep="", subset=["cells improved"]))

# One metric per protein
per_protein = pd.DataFrame({name: s["masked_pearson_r"] for name, s in scores.items()}).join(
    test.set_index("image_key")["gene_name"]).groupby("gene_name").mean()
display(per_protein.round(3))